<div style="
font-size:22px;
font-weight:600;
color:#1F2933;
">
Requirements:
<div style="
font-size:18px;
font-weight:600;
color:#1F2933;
"> 
Compatible with data_handling.py and fit_FMR_class.py (currently only Generalizing-complex-fits branch)
<div style="
font-size:22px;
font-weight:600;
color:#1F2933;
">
V Original
<div style="
font-size:18px;
font-weight:600;
color:#1F2933;
"> 

In [2]:
import h5py
import numpy as np 
import sys

import matplotlib.pyplot as plt

from pathlib import Path

sys.path.append(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis")
from data_handling import Analysis_FMR, FitStore
import fit_FMR_class as models
import complex_fits as fits
import fitting_functions as funcs
import resonance_functions as res_func
import calculators as calcs

import ipywidgets as widgets
from ipywidgets import interact

from scipy.optimize import curve_fit

<div style="
font-size:22px;
font-weight:600;
color:#1F2933;
">
Files Information:
<div style="
font-size:18px;
font-weight:600;
color:#1F2933;
">
Fill in the information about the files to be analyzed. Create a class entity for each file with Analysis_PSWS()

In [4]:
#Folders where file are. Keep the r to use raw string in Windows.
address1 = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\Sb2N2\inP\m10dBm\FMR_2026_08_20")

In [5]:
#File where all the sumarized data was stored. Normally it should be next to the raw data files.
file_name_1 = "Sb2N2_inP_m10dBm_LorentzFits_results_m10p0dBm.h5"

In [6]:
store_in_origin = True #True if you wnat to store fit results in orignal file, otherwise another h5f file is created in the same directory

In [7]:
val_type= 'freq'

In [8]:
newFile = address1/file_name_1

fits_storage = FitStore (newFile, val_type, group="Lorentzian fitting")

In [9]:
#Reloading parameters

In [10]:
data_L_fits = fits_storage.load_fit_results(newFile, group="Lorentzian fitting")

KeyError: "Unable to synchronously open object (object 'Lorentzian fitting' doesn't exist)"

In [ ]:
fits_storage.plot_fit_parameters_subplots(
    data_L_fits,
    x_axis="index",
    params=["df", "fo"],
    x_label= "Index"
)

In [ ]:
#We cut data if needed

In [ ]:
min_idx = 20
max_idx = 81
cut_data_L_fits = fits_storage.slice_dict(data_L_fits, min_idx, max_idx)

In [ ]:
#Plotting of fitted parameters

In [ ]:
fits_storage.plot_fit_parameters(
    cut_data_L_fits,
    x_axis="fo",
    params=["df"]
)

In [ ]:
#Fitting f. resonance vs H to the Kittel model

In [ ]:
H_data= cut_data_L_fits["field"]
f_data= cut_data_L_fits["fo"]

gamma_fix = True
gamma_fixed = 28.8

if gamma_fix:
    function = lambda H, Hx, Hy: res_func.kittel_general(H, gamma_fixed, Hx, Hy)
    p0 = [2.18, 0.04]  # initial guess: gamma ~ 28 GHz/T (electron-like)
    lower_bounds = [ 1.8, -1.0]
    upper_bounds = [2.5,  0.2]
else:
    function = res_func.kittel_general
    p0 = [28.8, 2.18, 0.04]  # initial guess: gamma ~ 28 GHz/T (electron-like)
    lower_bounds = [28.2, 1.8, -1.0]
    upper_bounds = [29.2,  2.5,  0.2]
    
popt, pcov = curve_fit( function,H_data,f_data,p0=p0,bounds=(lower_bounds, upper_bounds) )
perr = np.sqrt(np.diag(pcov))  # 1-sigma errors

H_fit = np.linspace(H_data.min(), H_data.max(), 200)
if gamma_fix:
    Hx_fit, Hy_fit = popt
    f_fit = res_func.kittel_general(H_fit, gamma_fixed, Hx_fit, Hy_fit)
    gamma_fit=gamma_fixed
else:
    gamma_fit, Hx_fit, Hy_fit = popt
    f_fit = res_func.kittel_general(H_fit, gamma_fit, Hx_fit, Hy_fit)

plt.scatter(H_data, f_data, label="Resoance f.")
plt.plot(H_fit, f_fit, label="Kittel fit", color="red")
plt.xlabel("H (T)")
plt.ylabel("Frequency (GHz)")
plt.legend()
plt.show()
print(f"gamma={gamma_fit}GHz/T, Hx={Hx_fit}T, Hy={Hy_fit}T")

In [ ]:
gamma = gamma_fit #GHz/T
inP = True #If measurment is inP or outP
Hx = Hx_fit #T
Hy = Hy_fit #T Only relevant for in-plane

dH = calcs.dH_from_df(2*cut_data_L_fits["df"], gamma, cut_data_L_fits["fo"], cut_data_L_fits["field"], Hx, Hy, inP)

In [ ]:
plt.scatter(cut_data_L_fits["fo"], dH)
plt.xlabel("f (GHz)")
plt.ylabel(r"$\Delta$H (T)")
plt.show()

In [ ]:
def linear(x, m, b):
    return m * x + b
    
popt, pcov = curve_fit(linear, cut_data_L_fits["fo"], dH)

   
slope, dH0 = popt
alpha = gamma * slope / 2
print(f"alpha={alpha}, dH0={dH0*1000}mT")

plt.rcParams.update({
    "font.size": 14,        # base font size
    "axes.labelsize": 16,   # x/y labels
    "axes.titlesize": 18,   # title
    "xtick.labelsize": 14,  # x tick labels
    "ytick.labelsize": 14,  # y tick labels
    "legend.fontsize": 14,
})

x_fit = np.linspace(np.min(cut_data_L_fits["fo"]), np.max(cut_data_L_fits["fo"]), 500)
y_fit = linear(x_fit, *popt)
plt.plot(x_fit, y_fit, label=f"Fit: y = {popt[0]:.3g}x + {popt[1]:.3g}", color="red")
plt.scatter(cut_data_L_fits["fo"], dH)
plt.xlabel("f (GHz)")
plt.ylabel(r"$\Delta$H (T)")
plt.show()